In [28]:
from pathlib import Path
import pandas as pd

def _loader(folder="./", loockback=100)-> pd.DataFrame:
    folder = Path(folder)
    stocks    = []

    for file in folder.glob("*.csv"):
        stock          =  file.stem
        df             =  pd.read_csv(file)
        df["Date"]     =  pd.to_datetime(df["Date"])            # Convert date
        df             =  (df.sort_values("Date"))             # Set index
        df["Stock"]    =  stock
        stocks.append(df)

    master                =  (pd.concat(stocks, ignore_index=True))
    all_dates             =  master["Date"].unique()
    all_stocks            =  master["Stock"].unique()
    template              = (pd.MultiIndex.from_product(
                             [all_dates, all_stocks],
                             names=["Date", "Stock"])
                            .to_frame(index=False)
                             )
    
    master              = template.merge(
                            master,
                            on=["Date", "Stock"],
                            how="left"
                            )
    
    ohlcv_cols          =  ["Open", "High", "Low", "Close", "Change", "Volume"]
    master              =  master.sort_values(["Stock", "Date"])
    master[ohlcv_cols]  =  (master.groupby("Stock")[ohlcv_cols].ffill())
    master[ohlcv_cols]  =  (master[ohlcv_cols]
                                .apply(lambda col: pd.to_numeric(col, errors="coerce")
                                .fillna(0)
                                 ))

    master             =  (master.sort_values(["Stock", "Date"])
                              .reset_index(drop=True)
                              .groupby("Stock")
                              .tail(loockback)
                              )
    master['Turnover']    = (master["Close"] * master["Volume"])
    master['Markert Cap'] = (0.231 * master["Turnover"])
    master['% Change']    = (master["Change"] * 100)
    master['Change']      = (master["Change"] * master["Close"])
    
    
    valid_cols         = ["Date", "Stock", "Open", "High", "Low", "Close", "Change", "% Change", "Volume", "Turnover", "Markert Cap"]
    return master[valid_cols]

In [29]:
df         =  _loader(folder="./Datasets", loockback=710)
latest     =  (df.sort_values(["Stock", "Date"])
              .reset_index(drop=True)
              .groupby("Stock")
              .tail(1)
              )

display(latest)

,Date,Stock,Open,High,Low,Close,Change,% Change,Volume,Turnover,Markert Cap
709,2026-03-19,AFRIPRISE,815.00,820.00,805.00,815.00,10.11,1.24,85306.00,69524390.00,16060134.09
1419,2026-03-19,CRDB,2880.00,2900.00,2810.00,2880.00,-10.08,-0.35,706654.00,2035163520.00,470122773.12
2129,2026-03-19,DCB,740.00,770.00,700.00,740.00,-37.96,-5.13,74611.00,55212140.00,12754004.34
2839,2026-03-19,DSE,6600.00,6600.00,6580.00,6600.00,0.00,0.00,1535.00,10131000.00,2340261.00
3549,2026-03-19,MBP,2600.00,2780.00,2580.00,2600.00,0.00,0.00,6042.00,15709200.00,3628825.20
4259,2026-03-19,MCB,1810.00,1810.00,1750.00,1810.00,94.66,5.23,5352.00,9687120.00,2237724.72
4969,2026-03-19,MKCB,4820.00,4970.00,4600.00,4820.00,112.79,2.34,4460.00,21497200.00,4965853.20
5679,2026-03-19,MUCOBA,900.00,1000.00,890.00,900.00,-85.95,-9.55,4094.00,3684600.00,851142.60
6389,2026-03-19,NICO,3660.00,3690.00,3510.00,3660.00,-9.88,-0.27,20518.00,75095880.00,17347148.28
7099,2026-03-19,PAL,570.00,640.00,555.00,570.00,-70.17,-12.31,17418.00,9928260.00,2293428.06


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12070 entries, 1759 to 41972
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         12070 non-null  datetime64[ns]
 1   Stock        12070 non-null  object        
 2   Open         12070 non-null  float64       
 3   High         12070 non-null  float64       
 4   Low          12070 non-null  float64       
 5   Close        12070 non-null  float64       
 6   Change       12070 non-null  float64       
 7   % Change     12070 non-null  float64       
 8   Volume       12070 non-null  float64       
 9   Turnover     12070 non-null  float64       
 10  Markert Cap  12070 non-null  float64       
dtypes: datetime64[ns](1), float64(9), object(1)
memory usage: 1.1+ MB


In [6]:
df.shape

(12070, 11)

In [7]:
df.columns

Index(['Date', 'Stock', 'Open', 'High', 'Low', 'Close', 'Change', '% Change',
       'Volume', 'Turnover', 'Markert Cap'],
      dtype='object')

In [26]:
df['Stock'].unique()

array(['AFRIPRISE', 'CRDB', 'DCB', 'DSE', 'MBP', 'MCB', 'MKCB', 'MUCOBA',
       'NICO', 'PAL', 'SWISS', 'TBL', 'TCC', 'TCCL', 'TOL', 'TTP', 'VODA'],
      dtype=object)

In [24]:
cols           =  ['Today', 'PD', 'DoD', 'WTD', 'PW', 'WoW', 'MTD', 'PM', 'MoM']
model          =  modeli(df, freqs=['W', 'M'])
turnover       =  _fmt(model.compute("Turnover"), cols, 1e6)
volume         =  _fmt(model.compute("Volume"), cols, 1e3)

In [22]:
display(turnover)

,Stock,Today,PD,DoD,DoD %,WTD,PW,WoW,WoW %,MTD,PM,MoM,MoM %
0,AFRIPRISE,69.52 M,69.52 M,0.00 M,0.00,210.77 M,299.62 M,-88.84 M,-29.65,"1,208.91 M","2,534.45 M","-1,325.54 M",-52.30
1,CRDB,"2,035.16 M","2,035.16 M",0.00 M,0.00,"8,743.07 M","35,062.08 M","-26,319.00 M",-75.06,"62,395.84 M","138,111.74 M","-75,715.89 M",-54.82
2,DCB,55.21 M,55.21 M,0.00 M,0.00,278.38 M,978.00 M,-699.62 M,-71.54,"2,666.02 M","1,814.24 M",851.78 M,46.95
3,DSE,10.13 M,10.13 M,0.00 M,0.00,60.33 M,157.55 M,-97.22 M,-61.71,431.30 M,706.61 M,-275.30 M,-38.96
4,MBP,15.71 M,15.71 M,0.00 M,0.00,55.03 M,136.31 M,-81.29 M,-59.63,328.83 M,947.08 M,-618.25 M,-65.28
5,MCB,9.69 M,9.69 M,0.00 M,0.00,52.65 M,115.39 M,-62.74 M,-54.37,582.18 M,380.20 M,201.98 M,53.12
6,MKCB,21.50 M,21.50 M,0.00 M,0.00,120.83 M,135.79 M,-14.96 M,-11.02,359.38 M,"1,107.12 M",-747.75 M,-67.54
7,MUCOBA,3.68 M,3.68 M,0.00 M,0.00,9.81 M,23.57 M,-13.75 M,-58.36,60.88 M,35.05 M,25.83 M,73.70
8,NICO,75.10 M,75.10 M,0.00 M,0.00,463.12 M,486.35 M,-23.22 M,-4.78,"1,427.77 M","1,626.59 M",-198.83 M,-12.22
9,PAL,9.93 M,9.93 M,0.00 M,0.00,32.31 M,173.57 M,-141.27 M,-81.39,276.13 M,48.80 M,227.33 M,465.83


In [25]:
display(turnover)

,Stock,Today,PD,DoD,DoD %,WTD,PW,WoW,WoW %,MTD,PM,MoM,MoM %
0,AFRIPRISE,69.52 M,69.52 M,0.00 M,0.00,210.77 M,299.62 M,-88.84 M,-29.65,"1,208.91 M","2,534.45 M","-1,325.54 M",-52.30
1,CRDB,"2,035.16 M","2,035.16 M",0.00 M,0.00,"8,743.07 M","35,062.08 M","-26,319.00 M",-75.06,"62,395.84 M","138,111.74 M","-75,715.89 M",-54.82
2,DCB,55.21 M,55.21 M,0.00 M,0.00,278.38 M,978.00 M,-699.62 M,-71.54,"2,666.02 M","1,814.24 M",851.78 M,46.95
3,DSE,10.13 M,10.13 M,0.00 M,0.00,60.33 M,157.55 M,-97.22 M,-61.71,431.30 M,706.61 M,-275.30 M,-38.96
4,MBP,15.71 M,15.71 M,0.00 M,0.00,55.03 M,136.31 M,-81.29 M,-59.63,328.83 M,947.08 M,-618.25 M,-65.28
5,MCB,9.69 M,9.69 M,0.00 M,0.00,52.65 M,115.39 M,-62.74 M,-54.37,582.18 M,380.20 M,201.98 M,53.12
6,MKCB,21.50 M,21.50 M,0.00 M,0.00,120.83 M,135.79 M,-14.96 M,-11.02,359.38 M,"1,107.12 M",-747.75 M,-67.54
7,MUCOBA,3.68 M,3.68 M,0.00 M,0.00,9.81 M,23.57 M,-13.75 M,-58.36,60.88 M,35.05 M,25.83 M,73.70
8,NICO,75.10 M,75.10 M,0.00 M,0.00,463.12 M,486.35 M,-23.22 M,-4.78,"1,427.77 M","1,626.59 M",-198.83 M,-12.22
9,PAL,9.93 M,9.93 M,0.00 M,0.00,32.31 M,173.57 M,-141.27 M,-81.39,276.13 M,48.80 M,227.33 M,465.83
